# Current systolic overlay: AXI/DMA smoke test

Targets the current `design_1.bit`: three 32-bit AXI DMA streams, Q8.8 payloads, and `matmul_top_0` AXI-Lite control. The first test is deliberately limited to one 16x16 tile.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

BITFILE = '/home/xilinx/pynq/overlays/matmul_systolic/design_1.bit'
overlay = Overlay(BITFILE)
dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1
matmul = overlay.matmul_top_0
print('Overlay loaded:', BITFILE)
print('IPs:', sorted(overlay.ip_dict))

In [ ]:
REG_CTRL = 0x00
REG_M = 0x10
REG_N = 0x18
REG_K = 0x20

def status():
    return {
        'A_MM2S': hex(dma_0.mmio.read(0x04)),
        'B_MM2S': hex(dma_1.mmio.read(0x04)),
        'C_S2MM': hex(dma_0.mmio.read(0x34)),
        'HLS_CTRL': hex(matmul.read(REG_CTRL)),
    }

def wait_idle(channel, name, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not channel.idle:
        if time.monotonic() >= deadline:
            raise TimeoutError(f'{name} timeout: {status()}')
        time.sleep(0.001)

print('Initial status:', status())

In [ ]:
M = N = K = 16
rng = np.random.default_rng(42)
A = rng.uniform(-1.0, 1.0, (M, K)).astype(np.float32)
B = rng.uniform(-1.0, 1.0, (K, N)).astype(np.float32)
A_q = np.rint(np.clip(A, -128, 127.99609375) * 256).astype(np.int16)
B_q = np.rint(np.clip(B, -128, 127.99609375) * 256).astype(np.int16)

# AXIS is 32 bits. Each word carries one sign-extended Q8.8 value.
a_buf = allocate((256,), dtype=np.int32)
b_buf = allocate((256,), dtype=np.int32)
c_buf = allocate((256,), dtype=np.int32)
np.copyto(a_buf, A_q.astype(np.int32).ravel())
np.copyto(b_buf, B_q.astype(np.int32).ravel())
c_buf[:] = 0
print('Physical addresses:', {
    'A': hex(a_buf.physical_address),
    'B': hex(b_buf.physical_address),
    'C': hex(c_buf.physical_address),
})

In [ ]:
try:
    matmul.write(REG_M, M)
    matmul.write(REG_N, N)
    matmul.write(REG_K, K)
    matmul.write(REG_CTRL, 1)
    print('HLS started:', status())

    # Arm the independent result DMA before the accelerator can emit C.
    dma_0.recvchannel.transfer(c_buf)
    dma_0.sendchannel.transfer(a_buf)
    dma_1.sendchannel.transfer(b_buf)
    print('All DMA channels started:', status())

    wait_idle(dma_0.sendchannel, 'A MM2S')
    print('A MM2S completed:', status())
    wait_idle(dma_1.sendchannel, 'B MM2S')
    print('B MM2S completed:', status())
    wait_idle(dma_0.recvchannel, 'C S2MM')
    print('C S2MM completed:', status())
finally:
    print('Final status:', status())

In [ ]:
C_hw = np.asarray(c_buf).reshape(16, 16).astype(np.float32) / 256.0
A_quantized = A_q.astype(np.float32) / 256.0
B_quantized = B_q.astype(np.float32) / 256.0
C_ref = A_quantized @ B_quantized
error = np.abs(C_hw - C_ref)
print('Maximum error:', float(error.max()))
print('Mean error:', float(error.mean()))
assert error.max() < 0.04, 'Hardware result exceeds Q8.8 tolerance'
C_hw

In [ ]:
a_buf.close()
b_buf.close()
c_buf.close()
print('Buffers released')